**Translation of the part of Hands-on [...] book NLP chapter from TF to Pytorch**

**Переводим часть главы по NLP из Hands-on [...] с TF на Pytorch**

#Sentiment

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
print(DEVICE)

cuda


In [ ]:
torch.manual_seed(42)

raw_datasets = load_dataset(
    "imdb", split=["train[:90%]", "train[90%:]", "test"]
)
raw_train_set, raw_valid_set, raw_test_set = raw_datasets
shuffled_train_set = raw_train_set.shuffle(seed=42)

train_set = DataLoader(shuffled_train_set, batch_size=32, shuffle=False, num_workers=2)
valid_set = DataLoader(raw_valid_set, batch_size=32, shuffle=False, num_workers=2)
test_set = DataLoader(raw_test_set, batch_size=32, shuffle=False, num_workers=2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
for i in range(4):
    item = raw_train_set[i]
    print(item["text"][:200], '[...]')
    print("Label:", item["label"])

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev [...]
Label: 0
"I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that  [...]
Label: 0
If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches  [...]
Label: 0
This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead.<br /><br />The film has two strong elements and those are, (1) the realistic acting (2) the impre [...]
Label: 0


In [ ]:
from transformers import PreTrainedTokenizerFast
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordLevelTrainer

In [ ]:
vocab_size = 1000

raw_tokenizer = Tokenizer(WordLevel(unk_token="<unk>"))
raw_tokenizer.pre_tokenizer = Whitespace()

trainer = WordLevelTrainer(
    vocab_size=vocab_size,
    special_tokens=["<pad>", "<unk>"]
)

def batch_iterator():
    for batch in train_set:
        yield batch["text"]

raw_tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=raw_tokenizer,
    pad_token="<pad>",
    unk_token="<unk>",
)

def text_pipeline(text_list):
    encoded = tokenizer(text_list)
    tokenized = encoded["input_ids"]

    lengths = [max(1, len(t)) for t in tokenized]
    max_len = max(lengths)
    pad_id = tokenizer.pad_token_id
    padded = [t + [pad_id] * (max_len - len(t)) for t in tokenized]

    return torch.tensor(padded, dtype=torch.long), \
           torch.tensor(lengths, dtype=torch.long)

In [ ]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, lengths):
        x = self.embedding(x)

        packed_x = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hn = self.gru(packed_x)

        # _, hn = self.gru(x)
        out = self.fc(hn[-1])
        return self.sigmoid(out)

In [ ]:
torch.manual_seed(42)
model = RNN(vocab_size, embed_size=128, hidden_size=128)
model.to(DEVICE)
optimizer = torch.optim.NAdam(model.parameters())
criterion = nn.BCELoss()

In [ ]:
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in train_set:
        inputs, lengths = text_pipeline(batch["text"])
        labels = torch.tensor(batch["label"], dtype=torch.float32).unsqueeze(1)
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(inputs, lengths)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * inputs.size(0)
        correct += ((outputs > 0.5) == labels).sum().item()
        total += inputs.size(0)

    print(f"Loss: {total_loss/total:.4f} - Accuracy: {correct/total:.4f}")

/tmp/ipykernel_3167/2880895222.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(batch["label"], dtype=torch.float32).unsqueeze(1)


Loss: 0.4801 - Accuracy: 0.7529
Loss: 0.2820 - Accuracy: 0.8812
Loss: 0.2117 - Accuracy: 0.9148
Loss: 0.1488 - Accuracy: 0.9452
Loss: 0.1141 - Accuracy: 0.9576


# Encoder-decoder

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

In [ ]:
url = "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
cache_dir = Path("datasets")
zip_path = cache_dir / "spa-eng.zip"

cache_dir.mkdir(parents=True, exist_ok=True)

if not zip_path.exists():
    print("Downloading...")
    urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(cache_dir)

text_file_path = cache_dir / "spa-eng" / "spa.txt"
text = text_file_path.read_text(encoding="utf-8")

print("Dataset loaded.")
print("\n".join(text.split("\n")[:3]))

Downloading...
Dataset loaded.
Go.	Ve.
Go.	Vete.
Go.	Vaya.


In [ ]:
import numpy as np

In [ ]:
text = text.replace("¡", "").replace("¿", "")
pairs = [line.split("\t") for line in text.splitlines()]
np.random.shuffle(pairs)
sentences_en, sentences_es = zip(*pairs)

In [ ]:
for i in range(3):
    print(sentences_en[i], "=>", sentences_es[i])

He can swim faster than any other boy in his class. => Él puede nadar más rápido que cualquier otro chico de su clase.
Look carefully. I'm going to show you how it's done. => Fíjate bien. Te voy a mostrar cómo se hace.
That's only a part of the problem. => Eso es sólo una parte del problema.


In [ ]:
import re
from collections import Counter

In [ ]:
vocab_size = 1000
# max_length = 25
max_length = 50

def clean_and_tokenize(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-zA-z0-9áéíóúüñ¿¡\s]", "", text)
    return text.split()

counter_en = Counter()
counter_es = Counter()

for sentence in sentences_en:
    counter_en.update(clean_and_tokenize(sentence))

for sentence in sentences_es:
    counter_es.update(clean_and_tokenize(sentence))

most_common_en = [word for word, _ in counter_en.most_common(vocab_size - 2)]
vocab_en = {word : idx + 2 for idx, word in enumerate(most_common_en)}
vocab_en["<pad>"] = 0
vocab_en["<unk>"] = 1

most_common_es = [word for word, _ in counter_es.most_common(vocab_size - 4)]
vocab_es = {word : idx + 4 for idx, word in enumerate(most_common_es)}
vocab_es["<pad>"] = 0
vocab_es["<unk>"] = 1
vocab_es["<sos>"] = 2
vocab_es["<eos>"] = 3

def en_to_ids(text, max_len=max_length):
    tokens = clean_and_tokenize(text)[:max_len]
    return [vocab_en.get(t, vocab_en["<unk>"]) for t in tokens]

def es_to_ids(text, max_len=max_length):
    tokens = clean_and_tokenize(text)[:max_len - 2]
    ids = [vocab_es["<sos>"]]
    ids += [vocab_es.get(t, vocab_es["<unk>"]) for t in tokens]
    ids.append(vocab_es["<eos>"])
    return ids

In [ ]:
[t for t, _ in sorted(vocab_en.items(), key=lambda item: item[1])][:10]

['<pad>', '<unk>', 'the', 'i', 'to', 'you', 'tom', 'a', 'is', 'he']

In [ ]:
train_en = sentences_en[:100_000]
valid_en = sentences_en[100_000:]

train_es = sentences_es[:100_000]
valid_es = sentences_es[100_000:]

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, en_sentences, es_sentences):
        self.en_sentences = en_sentences
        self.es_sentences = es_sentences

    def __len__(self):
        return len(self.en_sentences)

    def __getitem__(self, idx):
        en_text = self.en_sentences[idx]
        es_text = self.es_sentences[idx]

        encoder_input_ids = en_to_ids(en_text)
        es_tokens = clean_and_tokenize(es_text)[:max_length - 1]
        es_ids = [vocab_es.get(t, vocab_es["<unk>"]) for t in es_tokens]

        decoder_input_ids = [vocab_es["<sos>"]] + es_ids
        decoder_input_ids = decoder_input_ids[:max_length]

        target_ids = es_ids + [vocab_es["<eos>"]]
        target_ids = target_ids[:max_length]

        return {
            "encoder_input_ids" : encoder_input_ids,
            "decoder_input_ids" : decoder_input_ids,
            "target_ids" : target_ids
        }


In [ ]:
train_dataset = TranslationDataset(train_en, train_es)
valid_dataset = TranslationDataset(valid_en, valid_es)

In [ ]:
def translation_collate_fn(batch):
    enc_inputs = [item["encoder_input_ids"] for item in batch]
    dec_inputs = [item["decoder_input_ids"] for item in batch]
    targets = [item["target_ids"] for item in batch]

    padded_enc = [ids + [vocab_en["<pad>"]] * (max_length - len(ids)) for ids in enc_inputs]
    padded_dec = [ids + [vocab_es["<pad>"]] * (max_length - len(ids)) for ids in dec_inputs]
    padded_tar = [ids + [vocab_es["<pad>"]] * (max_length - len(ids)) for ids in targets]

    return (
        torch.tensor(padded_enc, dtype=torch.long),
        torch.tensor(padded_dec, dtype=torch.long),
        torch.tensor(padded_tar, dtype=torch.long)
    )

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64,
                          shuffle=True, collate_fn=translation_collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=64,
                          shuffle=False, collate_fn=translation_collate_fn)

In [ ]:
embed_size = 128

encoder_embedding_layer = nn.Embedding(num_embeddings=vocab_size,
                                       embedding_dim=embed_size,
                                       padding_idx=0)
decoder_embedding_layer = nn.Embedding(num_embeddings=vocab_size,
                                       embedding_dim=embed_size,
                                       padding_idx=0)

for enc_batch, dec_batch, target_batch in train_loader:
    encoder_embedding = encoder_embedding_layer(enc_batch)
    decoder_embedding = decoder_embedding_layer(dec_batch)

    print("enc shape", encoder_embedding.shape)
    print("dec shape", decoder_embedding.shape)
    break

enc shape torch.Size([64, 50, 128])
dec shape torch.Size([64, 50, 128])


In [ ]:
class Seq2SeqLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_size=512):
        super().__init__()
        self.encoder_embedding = nn.Embedding(num_embeddings=vocab_size,
                                       embedding_dim=embed_size,
                                       padding_idx=0)
        self.decoder_embedding = nn.Embedding(num_embeddings=vocab_size,
                                       embedding_dim=embed_size,
                                       padding_idx=0)
        self.encoder_lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.decoder_lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, encoder_input_ids, decoder_input_ids):
        enc_embed = self.encoder_embedding(encoder_input_ids)
        _, encoder_state = self.encoder_lstm(enc_embed)
        dec_embed = self.decoder_embedding(decoder_input_ids)
        decoder_out, _ = self.decoder_lstm(dec_embed, encoder_state)

        logits = self.fc(decoder_out)
        return logits

In [ ]:
vocab_size = 1000
model = Seq2SeqLSTM(vocab_size=vocab_size).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.NAdam(model.parameters(), lr=0.001)

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss, correct_tokens, total_tokens = 0, 0, 0

    for enc_batch, dec_batch, target_batch in train_loader:
        enc_batch = enc_batch.to(DEVICE)
        dec_batch = dec_batch.to(DEVICE)
        target_batch = target_batch.to(DEVICE)

        optimizer.zero_grad()

        logits = model(enc_batch, dec_batch)
        loss = criterion(logits.view(-1, vocab_size), target_batch.view(-1))
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        preds = logits.argmax(dim=-1)
        mask = (target_batch != 0)
        correct_tokens += ((preds == target_batch) & mask).sum().item()
        total_tokens += mask.sum().item()

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for enc_batch, dec_batch, target_batch in valid_loader:
            enc_batch = enc_batch.to(DEVICE)
            dec_batch = dec_batch.to(DEVICE)
            target_batch = target_batch.to(DEVICE)

            logits = model(enc_batch, dec_batch)
            loss = criterion(logits.view(-1, vocab_size), target_batch.view(-1))
            val_loss += loss.item()

            preds = logits.argmax(dim=-1)
            mask = (target_batch != 0)
            val_correct += ((preds == target_batch) & mask).sum().item()
            val_total += mask.sum().item()

    print(f"Epoch {epoch + 1}/{epochs}")
    print(f"Train Loss: {total_loss / len(train_loader):.4f}"
          f" | Train Acc: {correct_tokens / total_tokens:.4f}")
    print(f"Valid Loss: {val_loss / len(valid_loader):.4f}"
          f" | Valid Acc: {val_correct / val_total:.4f}")
    print("-" * 40)

Epoch 1/10
Train Loss: 2.7887 | Train Acc: 0.4410
Valid Loss: 1.9709 | Valid Acc: 0.5526
----------------------------------------
Epoch 2/10
Train Loss: 1.6384 | Train Acc: 0.6102
Valid Loss: 1.5189 | Valid Acc: 0.6318
----------------------------------------
Epoch 3/10
Train Loss: 1.2753 | Train Acc: 0.6791
Valid Loss: 1.3821 | Valid Acc: 0.6617
----------------------------------------
Epoch 4/10
Train Loss: 1.0713 | Train Acc: 0.7219
Valid Loss: 1.3263 | Valid Acc: 0.6723
----------------------------------------
Epoch 5/10
Train Loss: 0.9194 | Train Acc: 0.7548
Valid Loss: 1.3155 | Valid Acc: 0.6787
----------------------------------------
Epoch 6/10
Train Loss: 0.7950 | Train Acc: 0.7832
Valid Loss: 1.3248 | Valid Acc: 0.6786
----------------------------------------
Epoch 7/10
Train Loss: 0.6893 | Train Acc: 0.8081
Valid Loss: 1.3560 | Valid Acc: 0.6773
----------------------------------------
Epoch 8/10
Train Loss: 0.6012 | Train Acc: 0.8297
Valid Loss: 1.3941 | Valid Acc: 0.6777
-

In [ ]:
id_to_word_es = {id : word for word, id in vocab_es.items()}

def translate(sentence_en, model, max_length=50):
    model.eval()

    translation_words = []

    with torch.no_grad():
        en_ids = en_to_ids(sentence_en)
        padded_en = en_ids + [vocab_en["<pad>"]] * (max_length - len(en_ids))
        X = torch.tensor([padded_en], dtype=torch.long, device=DEVICE)

        for word_idx in range(max_length):
            dec_ids = [vocab_es["<sos>"]] + \
             [vocab_es.get(w, vocab_es["<unk>"]) for w in translation_words]
            padded_dec = dec_ids + [vocab_es["<pad>"]] * (max_length - len(en_ids))
            X_dec = torch.tensor([padded_dec], dtype=torch.long, device=DEVICE)

            logits = model(X, X_dec)
            current_logits = logits[0, word_idx]
            pred_id = current_logits.argmax().item()
            pred = id_to_word_es.get(pred_id, "<unk>")

            if pred == "<eos>":
                break

            translation_words.append(pred)

    return " ".join(translation_words)

In [ ]:
translate("I like soccer")

'me gusta el fútbol'

## RNN + attention

In [ ]:
import torch.nn.functional as F

In [ ]:
class AttentionSeq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_size=256):
        super().__init__()
        self.encoder_embedding = nn.Embedding(num_embeddings=vocab_size,
                                       embedding_dim=embed_size,
                                       padding_idx=0)
        self.decoder_embedding = nn.Embedding(num_embeddings=vocab_size,
                                       embedding_dim=embed_size,
                                       padding_idx=0)

        self.encoder_lstm = nn.LSTM(embed_size, hidden_size, batch_first=True,
                                    bidirectional=True)

        encoder_out_size = hidden_size * 2
        self.w_h = nn.Linear(encoder_out_size, hidden_size)
        self.w_c = nn.Linear(encoder_out_size, hidden_size)

        self.decoder_lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

        self.att_matrix = nn.Linear(hidden_size, encoder_out_size, bias=False)
        self.attention_combine = nn.Linear(encoder_out_size + hidden_size,
                                           hidden_size)

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, encoder_input_ids, decoder_input_ids):
        enc_embed = self.encoder_embedding(encoder_input_ids)
        enc_out, (h_n, c_n) = self.encoder_lstm(enc_embed)

        h_concat = torch.cat((h_n[0], h_n[1]), dim=-1)
        c_concat = torch.cat((c_n[0], c_n[1]), dim=-1)

        h_final = torch.tanh(self.w_h(h_concat)).unsqueeze(0)
        c_final = torch.tanh(self.w_h(c_concat)).unsqueeze(0)

        dec_embed = self.decoder_embedding(decoder_input_ids)
        decoder_out, _ = self.decoder_lstm(dec_embed, (h_final, c_final))

        # Attention

        dec_out_proj = self.att_matrix(decoder_out)
        scores = torch.bmm(dec_out_proj, enc_out.transpose(1, 2))

        att_weights = F.softmax(scores, dim=-1)

        context_vec = torch.bmm(att_weights, enc_out)
        combined = torch.cat((context_vec, decoder_out), dim=-1)
        att_out = torch.tanh(self.attention_combine(combined))

        logits = self.fc(att_out)
        return logits

In [ ]:
vocab_size = 1000
model = AttentionSeq2Seq(vocab_size=vocab_size).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.NAdam(model.parameters(), lr=0.001)

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss, correct_tokens, total_tokens = 0, 0, 0

    for enc_batch, dec_batch, target_batch in train_loader:
        enc_batch = enc_batch.to(DEVICE)
        dec_batch = dec_batch.to(DEVICE)
        target_batch = target_batch.to(DEVICE)

        optimizer.zero_grad()

        logits = model(enc_batch, dec_batch)
        loss = criterion(logits.view(-1, vocab_size), target_batch.view(-1))
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        preds = logits.argmax(dim=-1)
        mask = (target_batch != 0)
        correct_tokens += ((preds == target_batch) & mask).sum().item()
        total_tokens += mask.sum().item()

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for enc_batch, dec_batch, target_batch in valid_loader:
            enc_batch = enc_batch.to(DEVICE)
            dec_batch = dec_batch.to(DEVICE)
            target_batch = target_batch.to(DEVICE)

            logits = model(enc_batch, dec_batch)
            loss = criterion(logits.view(-1, vocab_size), target_batch.view(-1))
            val_loss += loss.item()

            preds = logits.argmax(dim=-1)
            mask = (target_batch != 0)
            val_correct += ((preds == target_batch) & mask).sum().item()
            val_total += mask.sum().item()

    print(f"Epoch {epoch + 1}/{epochs}")
    print(f"Train Loss: {total_loss / len(train_loader):.4f}"
          f" | Train Acc: {correct_tokens / total_tokens:.4f}")
    print(f"Valid Loss: {val_loss / len(valid_loader):.4f}"
          f" | Valid Acc: {val_correct / val_total:.4f}")
    print("-" * 40)

Epoch 1/10
Train Loss: 2.3639 | Train Acc: 0.5320
Valid Loss: 1.5654 | Valid Acc: 0.6492
----------------------------------------
Epoch 2/10
Train Loss: 1.3490 | Train Acc: 0.6865
Valid Loss: 1.2962 | Valid Acc: 0.6950
----------------------------------------
Epoch 3/10
Train Loss: 1.1283 | Train Acc: 0.7254
Valid Loss: 1.2024 | Valid Acc: 0.7139
----------------------------------------
Epoch 4/10
Train Loss: 0.9998 | Train Acc: 0.7492
Valid Loss: 1.1572 | Valid Acc: 0.7208
----------------------------------------
Epoch 5/10
Train Loss: 0.9009 | Train Acc: 0.7686
Valid Loss: 1.1462 | Valid Acc: 0.7261
----------------------------------------
Epoch 6/10
Train Loss: 0.8209 | Train Acc: 0.7838
Valid Loss: 1.1408 | Valid Acc: 0.7270
----------------------------------------
Epoch 7/10
Train Loss: 0.7512 | Train Acc: 0.7979
Valid Loss: 1.1477 | Valid Acc: 0.7297
----------------------------------------
Epoch 8/10
Train Loss: 0.6881 | Train Acc: 0.8114
Valid Loss: 1.1641 | Valid Acc: 0.7280
-

In [ ]:
translate("I like soccer", model)

'me gusta el fútbol'

In [ ]:
translate("I like soccer and also going to the beach", model)

'me gusta el fútbol y también voy a la playa'

In [ ]:
translate("I like basketball", model)

'me gusta el <unk>'

#Transformer

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, max_length, embed_size):
        super().__init__()
        assert embed_size % 2 == 0, "embed_size must be even"

        p, i = np.meshgrid(np.arange(max_length), 2 * np.arange(embed_size // 2))
        pos_embed = np.empty((1, max_length, embed_size))
        pos_embed[0, :, ::2] = np.sin(p / 10_000 ** (i / embed_size)).T
        pos_embed[0, :, 1::2] = np.cos(p / 10_000 ** (i / embed_size)).T

        pos_encoding_tensor = torch.tensor(pos_embed, dtype=torch.float32)
        self.register_buffer("pos_encodings", pos_encoding_tensor)

    def forward(self, inputs):
        batch_max_length = inputs.size(1)
        return inputs + self.pos_encodings[:, :batch_max_length]

In [ ]:
max_length = 50
embed_size = 128

pos_embed_lyr = PositionalEncoding(max_length, embed_size)
pos_embed_lyr = pos_embed_lyr.to(DEVICE)

enc_embeddings = torch.randn(64, 35, embed_size, device=DEVICE)
dec_embeddings = torch.randn(64, 20, embed_size, device=DEVICE)

enc_in = pos_embed_lyr(enc_embeddings)
dec_in = pos_embed_lyr(dec_embeddings)

print(enc_in.shape)
print(dec_in.shape)


torch.Size([64, 35, 128])
torch.Size([64, 20, 128])


In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, smbed_size, num_heads, n_units, dropout_rate):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim=embed_size, num_heads=num_heads,
                                        dropout=dropout_rate, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.fnn = nn.Sequential(
            nn.Linear(embed_size, n_units),
            nn.ReLU(),
            nn.Linear(n_units, embed_size),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x, mask):
        att_out, _ = self.mha(x, x, x, attn_mask=mask)
        x = self.norm1(x + att_out)

        fnn_out = self.fnn(x)
        x = self.norm2(x + fnn_out)
        return x

class DecoderLayer(nn.Module):
    def __init__(self, smbed_size, num_heads, n_units, dropout_rate):
        super().__init__()
        self.mha1 = nn.MultiheadAttention(embed_dim=embed_size, num_heads=num_heads,
                                        dropout=dropout_rate, batch_first=True)
        self.mha2 = nn.MultiheadAttention(embed_dim=embed_size, num_heads=num_heads,
                                        dropout=dropout_rate, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)
        self.norm3 = nn.LayerNorm(embed_size)

        self.fnn = nn.Sequential(
            nn.Linear(embed_size, n_units),
            nn.ReLU(),
            nn.Linear(n_units, embed_size),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x, enc_out, causal_mask, enc_mask):
        att_out1, _ = self.mha1(x, x, x, attn_mask=causal_mask)
        x = self.norm1(x + att_out1)

        att_out2, _ = self.mha2(x, enc_out, enc_out, attn_mask=enc_mask)
        x = self.norm2(x + att_out2)

        fnn_out = self.fnn(x)
        x = self.norm3(x + fnn_out)
        return x

In [ ]:
class FullTransformer(nn.Module):
    def __init__(self, vocab_size, max_length=50, embed_size=128,
                 hidden_size=256, N=2, num_heads=8, n_units=128,
                 dropout_rate=0.1):

        super().__init__()
        self.encoder_embedding = nn.Embedding(num_embeddings=vocab_size,
                                       embedding_dim=embed_size,
                                       padding_idx=0)
        self.decoder_embedding = nn.Embedding(num_embeddings=vocab_size,
                                       embedding_dim=embed_size,
                                       padding_idx=0)

        self.pos_encoder = PositionalEncoding(max_length, embed_size)

        self.encoder_layers = nn.ModuleList(
            [EncoderLayer(embed_size, num_heads, n_units, dropout_rate)
            for _ in range(N)])
        self.decoder_layers = nn.ModuleList(
            [DecoderLayer(embed_size, num_heads, n_units, dropout_rate)
            for _ in range(N)])

        self.fc = nn.Linear(embed_size, vocab_size)
        self.num_heads = num_heads

    def generate_causal_mask(self, seq_len, device):
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device),
                          diagonal=1).bool()
        return mask

    def forward(self, encoder_input_ids, decoder_input_ids):

        device = encoder_input_ids.device

        enc_pad_mask = (encoder_input_ids == 0).unsqueeze(1).repeat(
            1, encoder_input_ids.size(1), 1)
        enc_pad_mask = enc_pad_mask.repeat_interleave(self.num_heads, dim=0)

        dec_pad_mask = (decoder_input_ids == 0).unsqueeze(1).repeat(
            1, decoder_input_ids.size(1), 1)

        cross_pad_mask = (encoder_input_ids == 0).unsqueeze(1).repeat(
            1, decoder_input_ids.size(1), 1)
        cross_pad_mask = cross_pad_mask.repeat_interleave(self.num_heads, dim=0)

        causal_mask = self.generate_causal_mask(decoder_input_ids.size(1),
                                                device)

        combined_dec_mask = causal_mask.unsqueeze(0) | dec_pad_mask

        combined_dec_mask = combined_dec_mask.repeat_interleave(self.num_heads, dim=0)


        Z = self.encoder_embedding(encoder_input_ids)
        Z = self.pos_encoder(Z)
        for lyr in self.encoder_layers:
            Z = lyr(Z, enc_pad_mask)

        enc_out = Z

        Z_dec = self.decoder_embedding(decoder_input_ids)
        Z_dec = self.pos_encoder(Z_dec)
        for lyr in self.decoder_layers:
            Z_dec = lyr(Z_dec, enc_out, combined_dec_mask, cross_pad_mask)

        logits = self.fc(Z_dec)
        return logits

In [ ]:
vocab_size = 1000
model = FullTransformer(vocab_size=vocab_size).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.NAdam(model.parameters(), lr=0.001)

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss, correct_tokens, total_tokens = 0, 0, 0

    for enc_batch, dec_batch, target_batch in train_loader:
        enc_batch = enc_batch.to(DEVICE)
        dec_batch = dec_batch.to(DEVICE)
        target_batch = target_batch.to(DEVICE)

        optimizer.zero_grad()

        logits = model(enc_batch, dec_batch)
        loss = criterion(logits.view(-1, vocab_size), target_batch.view(-1))
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        preds = logits.argmax(dim=-1)
        mask = (target_batch != 0)
        correct_tokens += ((preds == target_batch) & mask).sum().item()
        total_tokens += mask.sum().item()

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for enc_batch, dec_batch, target_batch in valid_loader:
            enc_batch = enc_batch.to(DEVICE)
            dec_batch = dec_batch.to(DEVICE)
            target_batch = target_batch.to(DEVICE)

            logits = model(enc_batch, dec_batch)
            loss = criterion(logits.view(-1, vocab_size), target_batch.view(-1))
            val_loss += loss.item()

            preds = logits.argmax(dim=-1)
            mask = (target_batch != 0)
            val_correct += ((preds == target_batch) & mask).sum().item()
            val_total += mask.sum().item()

    print(f"Epoch {epoch + 1}/{epochs}")
    print(f"Train Loss: {total_loss / len(train_loader):.4f}"
          f" | Train Acc: {correct_tokens / total_tokens:.4f}")
    print(f"Valid Loss: {val_loss / len(valid_loader):.4f}"
          f" | Valid Acc: {val_correct / val_total:.4f}")
    print("-" * 40)

Epoch 1/10
Train Loss: 2.2275 | Train Acc: 0.5355
Valid Loss: 1.5505 | Valid Acc: 0.6361
----------------------------------------
Epoch 2/10
Train Loss: 1.4555 | Train Acc: 0.6495
Valid Loss: 1.3590 | Valid Acc: 0.6719
----------------------------------------
Epoch 3/10
Train Loss: 1.2999 | Train Acc: 0.6783
Valid Loss: 1.2978 | Valid Acc: 0.6824
----------------------------------------
Epoch 4/10
Train Loss: 1.2139 | Train Acc: 0.6948
Valid Loss: 1.2538 | Valid Acc: 0.6946
----------------------------------------
Epoch 5/10
Train Loss: 1.1533 | Train Acc: 0.7059
Valid Loss: 1.2251 | Valid Acc: 0.7010
----------------------------------------
Epoch 6/10
Train Loss: 1.1068 | Train Acc: 0.7153
Valid Loss: 1.1963 | Valid Acc: 0.7068
----------------------------------------
Epoch 7/10
Train Loss: 1.0706 | Train Acc: 0.7220
Valid Loss: 1.1837 | Valid Acc: 0.7076
----------------------------------------
Epoch 8/10
Train Loss: 1.0406 | Train Acc: 0.7281
Valid Loss: 1.1739 | Valid Acc: 0.7119
-

In [ ]:
def translate_transformer(sentence_en, model, max_length=50):
    model.eval()
    with torch.no_grad():
        en_ids = en_to_ids(sentence_en)
        padded_en = en_ids + [vocab_en["<pad>"]] * (max_length - len(en_ids))
        X = torch.tensor([padded_en], dtype=torch.long, device=DEVICE)
        X_dec = torch.tensor([[vocab_es["<sos>"]]], dtype=torch.long, device=DEVICE)
        translation_words = []

        for _ in range(max_length):
            logits = model(X, X_dec)
            next_token_ligits = logits[0, -1, :]
            pred_id = next_token_ligits.argmax().item()
            pred = id_to_word_es.get(pred_id, "<unk>")

            if pred == "<eos>":
                break

            translation_words.append(pred)
            new_token_tensor = torch.tensor([[pred_id]],
                                            dtype=torch.long, device=DEVICE)
            X_dec = torch.cat((X_dec, new_token_tensor), dim=1)

    return " ".join(translation_words)

In [ ]:
translate_transformer("I like soccer and also going to the beach", model)

'me gusta el fútbol y también ir a la playa'

In [ ]:
translate_transformer("I like soccer", model)

'me gusta el fútbol'